In [1]:
from collections import deque
from typing import List, Optional, Tuple

# 0 la o trong
State = Tuple[int, ...]
GOAL_STATE: State = (1, 2, 3, 4, 5, 6, 7, 8, 0)

# Quy uoc: UP/DOWN/LEFT/RIGHT la huong di chuyen cua o trong 0
MOVES = {
    "UP": (-1, 0, "len"),
    "DOWN": (1, 0, "xuong"),
    "LEFT": (0, -1, "trai"),
    "RIGHT": (0, 1, "phai"),
}


def print_matrix(state: State) -> None:
    for i in range(0, 9, 3):
        print(state[i:i + 3])
    print()


def valid_moves(state: State) -> List[str]:
    blank = state.index(0)
    row, col = divmod(blank, 3)
    actions = []

    for action, (dr, dc, _) in MOVES.items():
        new_row = row + dr
        new_col = col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            actions.append(action)

    return actions


def apply_move(state: State, action: str) -> State:
    blank = state.index(0)
    row, col = divmod(blank, 3)
    dr, dc, _ = MOVES[action]
    target = (row + dr) * 3 + (col + dc)

    new_state = list(state)
    new_state[blank], new_state[target] = new_state[target], new_state[blank]
    return tuple(new_state)


def is_solvable(state: State) -> bool:
    numbers = [x for x in state if x != 0]
    inversions = 0

    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] > numbers[j]:
                inversions += 1

    return inversions % 2 == 0


class Puzzle8Environment:
    def __init__(self, initial_state: State):
        self.state = initial_state

    def percept(self) -> State:
        return self.state

    def step(self, action: str) -> State:
        self.state = apply_move(self.state, action)
        return self.state


class ModelBasedReflexAgent:
    def __init__(self, goal_state: State):
        self.goal_state = goal_state
        self.model: Optional[State] = None
        self.visited = set()
        self.plan: List[str] = []

    def update_model(self, percept: State) -> None:
        self.model = percept
        self.visited.add(percept)

    def find_plan_by_bfs(self) -> List[str]:
        queue = deque([(self.model, [])])
        seen = {self.model}

        while queue:
            state, path = queue.popleft()

            if state == self.goal_state:
                return path

            for action in valid_moves(state):
                new_state = apply_move(state, action)
                if new_state not in seen:
                    seen.add(new_state)
                    queue.append((new_state, path + [action]))

        return []

    def choose_action(self) -> Optional[str]:
        if self.model is None or self.model == self.goal_state:
            return None

        if not self.plan:
            self.plan = self.find_plan_by_bfs()

        if not self.plan:
            return None

        return self.plan.pop(0)


def solve_puzzle(initial_state: State, max_steps: int = 50) -> None:
    if not is_solvable(initial_state):
        print("Trang thai nay khong giai duoc.")
        return

    env = Puzzle8Environment(initial_state)
    agent = ModelBasedReflexAgent(GOAL_STATE)
    actions_done = []

    print("Quy uoc: 0 la o trong; action la huong di chuyen cua o trong.")
    print("Trang thai ban dau:")
    print_matrix(env.percept())

    for step in range(1, max_steps + 1):
        percept = env.percept()
        agent.update_model(percept)
        action = agent.choose_action()

        if action is None:
            break

        new_state = env.step(action)
        actions_done.append(action)
        print(f"Buoc {step}: di chuyen o trong sang {MOVES[action][2].upper()} ({action})")
        print_matrix(new_state)

        if new_state == GOAL_STATE:
            print("Da dat trang thai dich!")
            break

    print("Cac buoc di:", " -> ".join(actions_done))
    print("So buoc:", len(actions_done))


# Ban co the doi trang thai ban dau tai day
initial = (1, 2, 3,
           4, 0, 5,
           6, 7, 8)

solve_puzzle(initial)


Quy uoc: 0 la o trong; action la huong di chuyen cua o trong.
Trang thai ban dau:
(1, 2, 3)
(4, 0, 5)
(6, 7, 8)

Buoc 1: di chuyen o trong sang PHAI (RIGHT)
(1, 2, 3)
(4, 5, 0)
(6, 7, 8)

Buoc 2: di chuyen o trong sang XUONG (DOWN)
(1, 2, 3)
(4, 5, 8)
(6, 7, 0)

Buoc 3: di chuyen o trong sang TRAI (LEFT)
(1, 2, 3)
(4, 5, 8)
(6, 0, 7)

Buoc 4: di chuyen o trong sang TRAI (LEFT)
(1, 2, 3)
(4, 5, 8)
(0, 6, 7)

Buoc 5: di chuyen o trong sang LEN (UP)
(1, 2, 3)
(0, 5, 8)
(4, 6, 7)

Buoc 6: di chuyen o trong sang PHAI (RIGHT)
(1, 2, 3)
(5, 0, 8)
(4, 6, 7)

Buoc 7: di chuyen o trong sang XUONG (DOWN)
(1, 2, 3)
(5, 6, 8)
(4, 0, 7)

Buoc 8: di chuyen o trong sang PHAI (RIGHT)
(1, 2, 3)
(5, 6, 8)
(4, 7, 0)

Buoc 9: di chuyen o trong sang LEN (UP)
(1, 2, 3)
(5, 6, 0)
(4, 7, 8)

Buoc 10: di chuyen o trong sang TRAI (LEFT)
(1, 2, 3)
(5, 0, 6)
(4, 7, 8)

Buoc 11: di chuyen o trong sang TRAI (LEFT)
(1, 2, 3)
(0, 5, 6)
(4, 7, 8)

Buoc 12: di chuyen o trong sang XUONG (DOWN)
(1, 2, 3)
(4, 5, 6)
(0, 7, 